# Training 12 · Speech enhancer pre-training (E3)

A 2.6 M-parameter mask-based enhancer trained on VoiceBank-DEMAND with a supervised loss; 3 seeds, about 45 min each.
This is the starting point of every GRPO-trained enhancer in `training/13_enhancer_grpo.ipynb`.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = ["se_pretrain_s0", "se_pretrain_s1", "se_pretrain_s2"]

In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: se_pretrain
task: se_pretrain
phase: main
seed: 0
run_name: se_pretrain_s0__2026-09-15_16-10-03
model:
  n_fft: 512
  hop: 128
  hidden: 384
  rnn_hidden: 256
  rnn_layers: 2
  bidirectional: true
  conv_kernel: 3
  mask_max: 2.0
  init_log_std: -1.0
  min_log_std: -5.0
  max_log_std: 1.0
data:
  crop_seconds: 3.0
  num_workers: 8
  metric_workers: 8
  max_train_utts: null
  max_val_utts: null
  subset_seed: 0
pretrain:
  steps: 50000
  batch_size: 32
  lr: 0.001
  weight_decay: 1.0e-05
  warmup_steps: 1000
  cosine: true
  final_lr_frac: 0.02
  grad_clip: 5.0
  w_sisdr: 1.0
  w_mag: 10.0
  mag_compress: 0.3
  log_every: 50
  eval_every: 2500
  report_noisy: true
wandb:
  entity: alialavi
  mode: online
  group: se_pretrain



In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for se_pretrain_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_se_pretrain_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key se_pretrain_s0



## Retrained vs original

In [6]:
released = repro.fetch_results()
rows = {}
for k in KEYS:
    orig = sorted(released.glob(f"{k}__*"))
    for src, d in (("original", orig[-1] if orig else None), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "results" / "val_final_summary.csv" if d is not None else None
        if f is not None and f.exists():
            rows[(k, src)] = pd.read_csv(f).mean(numeric_only=True)
pd.DataFrame(rows).T.round(3) if rows else "no runs found"

,,value
se_pretrain_s0,original,3575.262
se_pretrain_s1,original,2860.968
se_pretrain_s2,original,3218.114
